# Đo ngân sách GPU DeepWeeds — upload một file
Chọn Runtime → Change runtime type → GPU (T4 nếu có), rồi Runtime → Run all.
Notebook có sẵn script, tự tải ảnh/CSV fold 0 và kiểm tra MD5.
Chỉ đo một epoch train/val của mỗi backbone, không chạy test hay thực hiện toàn bộ bài lab.
GPU_HOURS=None nghĩa là chưa nhập hạn mức; kế hoạch chưa được chốt.


In [ ]:
import sys, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'pandas', 'pillow'], check=True)
import torch
assert torch.cuda.is_available(), 'Bật GPU trong cài đặt phiên chạy.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)

In [ ]:
# Tự đặt đường dẫn cho Colab hoặc Kaggle, không cần upload repo.
# Ưu tiên thư mục Colab; Colab cũng có thể chứa /kaggle.
BASE_DIR = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working')
REPO_DIR = BASE_DIR / 'gpu_budget_lab'
IMAGES_DIR = BASE_DIR / 'data/images'
LABELS_DIR = BASE_DIR / 'data/labels'
OUTPUT_DIR = BASE_DIR / 'gpu_budget'
GPU_HOURS = None  # Điền hạn mức giờ GPU thật nếu đã biết.
BATCH_SIZE = 32
EPOCHS = 10
SELECTED = ['resnet50']  # Giả định tính giờ, chưa phải model thắng trên val.
FINAL_BACKBONE = SELECTED[0]
(REPO_DIR / 'tools').mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Code đo epoch được nhúng sẵn trong notebook.
(REPO_DIR / 'tools/gpu_budget.py').write_text('"""Measure one real DeepWeeds train/val epoch; extrapolate a GPU budget.\n\nThis diagnostic does not implement the lab pipeline or select models by quality.\nIt never evaluates test images. Import ML dependencies only in `measure`.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport json\nimport math\nimport platform\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nBACKBONES = ["resnet50", "resnext50_32x4d", "convnext_tiny",\n             "deit_small_patch16_224", "efficientnet_b0"]\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\ndef load_fold(labels_dir, images_dir):\n    splits = {}\n    for split in ("train", "val", "test"):\n        with (Path(labels_dir) / f"{split}_subset0.csv").open(encoding="utf-8-sig", newline="") as f:\n            rows = list(csv.DictReader(f))\n        names = [r["Filename"] for r in rows]\n        if not rows or len(names) != len(set(names)):\n            raise ValueError(f"Empty split or duplicate filenames: {split}")\n        for r in rows:\n            if not 0 <= int(r["Label"]) < 9:\n                raise ValueError(f"Invalid Label: {r}")\n            if not (Path(images_dir) / r["Filename"]).is_file():\n                raise ValueError(f"Missing image: {r[\'Filename\']}")\n        splits[split] = rows\n    sets = {s: {r["Filename"] for r in rows} for s, rows in splits.items()}\n    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):\n        if sets[a] & sets[b]:\n            raise ValueError(f"Overlap: {a}/{b}")\n    if len(set.union(*sets.values())) != 17509:\n        raise ValueError("Fold 0 must contain 17,509 distinct filenames.")\n    for split, rows in splits.items():\n        expected = 0.6 if split == "train" else 0.2\n        if abs(len(rows) / 17509 - expected) > 0.01:\n            raise ValueError(f"Unexpected split size: {split}={len(rows)}")\n    return splits\n\n\ndef measure(args):\n    import numpy as np\n    import torch\n    import timm\n    import torchvision\n    from PIL import Image\n    from torch.utils.data import DataLoader, Dataset\n    from torchvision import transforms as T\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Enable a GPU in Colab/Kaggle before measuring.")\n    splits = load_fold(args.labels_dir, args.images_dir)\n    if args.batch_size > len(splits["train"]):\n        raise ValueError("Batch size cannot exceed the train split size.")\n    if len(set(args.backbones)) != len(args.backbones):\n        raise ValueError("Backbone names must be distinct.")\n    # Local Dataset classes can use Linux fork workers; Windows uses spawn.\n    workers = 0 if platform.system() == "Windows" else args.workers\n\n    class Images(Dataset):\n        def __init__(self, rows, transform):\n            self.rows, self.transform = rows, transform\n\n        def __len__(self):\n            return len(self.rows)\n\n        def __getitem__(self, index):\n            row = self.rows[index]\n            with Image.open(Path(args.images_dir) / row["Filename"]) as img:\n                x = self.transform(img.convert("RGB"))\n            return x, int(row["Label"])\n\n    report = {\n        "purpose": "One-epoch timing only; no final lab results or test inference",\n        "gpu": torch.cuda.get_device_name(0), "python": platform.python_version(),\n        "torch": torch.__version__, "timm": timm.__version__,\n        "torchvision": torchvision.__version__, "seed": args.seed,\n        "batch_size": args.batch_size, "img_size": args.img_size,\n        "num_workers": workers, "amp": True,\n        "planned_epochs": args.epochs,\n        "split_sizes": {s: len(rows) for s, rows in splits.items()},\n        "per_class": {s: [sum(int(r["Label"]) == c for r in rows) for c in range(9)]\n                      for s, rows in splits.items()},\n        "recipe": {"optimizer": "AdamW", "lr_backbone": 1e-4, "lr_head": 1e-3,\n                   "weight_decay": 0.05, "norm_bias_decay": 0,\n                   "loss": "CE", "init": "ImageNet finetune",\n                   "scheduler": "1 epoch linear warmup, then cosine"},\n        "models": [],\n    }\n    print("GPU:", report["gpu"], "| splits:", report["split_sizes"], flush=True)\n    for name in args.backbones:\n        # Same seed, batch size, recipe and split for every architecture.\n        random.seed(args.seed)\n        np.random.seed(args.seed)\n        torch.manual_seed(args.seed)\n        torch.cuda.manual_seed_all(args.seed)\n        torch.backends.cudnn.benchmark = False\n        setup_start = time.perf_counter()\n        kwargs = {"img_size": args.img_size} if name.startswith(("vit_", "deit_", "swin_")) else {}\n        model = timm.create_model(name, pretrained=True, num_classes=9, **kwargs).cuda()\n        cfg = dict(model.pretrained_cfg)\n        norm = T.Normalize(cfg.get("mean", (0.485, 0.456, 0.406)),\n                           cfg.get("std", (0.229, 0.224, 0.225)))\n        interp = T.InterpolationMode.BICUBIC if cfg.get("interpolation") == "bicubic" else T.InterpolationMode.BILINEAR\n        train_tf = T.Compose([T.RandomResizedCrop(args.img_size, interpolation=interp),\n                              T.RandomHorizontalFlip(), T.ToTensor(), norm])\n        val_tf = T.Compose([T.Resize(round(args.img_size * 256 / 224), interpolation=interp),\n                            T.CenterCrop(args.img_size), T.ToTensor(), norm])\n        loaders = {s: DataLoader(Images(splits[s], transform), batch_size=args.batch_size,\n                                shuffle=(s == "train"), drop_last=(s == "train"),\n                                num_workers=workers, pin_memory=True)\n                   for s, transform in (("train", train_tf), ("val", val_tf))}\n        head_ids = {id(p) for p in model.get_classifier().parameters()}\n        groups = {}\n        for p in model.parameters():\n            key = (1e-3 if id(p) in head_ids else 1e-4, 0.05 if p.ndim > 1 else 0.0)\n            groups.setdefault(key, []).append(p)\n        optimizer = torch.optim.AdamW([{"params": p, "lr": lr, "weight_decay": wd}\n                                      for (lr, wd), p in groups.items()])\n        steps = len(loaders["train"])\n        def lr_factor(step):\n            if step < steps:\n                return (step + 1) / steps\n            progress = min(1.0, (step - steps) / max(1, (args.epochs - 1) * steps))\n            return 0.5 * (1 + math.cos(math.pi * progress))\n        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)\n        scaler = torch.amp.GradScaler("cuda")\n        torch.cuda.synchronize()\n        setup_s = time.perf_counter() - setup_start\n        torch.cuda.reset_peak_memory_stats()\n        model.train()\n        start = time.perf_counter()\n        train_n = 0\n        for x, y in loaders["train"]:\n            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type="cuda", dtype=torch.float16):\n                loss = torch.nn.functional.cross_entropy(model(x), y)\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            train_n += len(y)\n        torch.cuda.synchronize()\n        train_s = time.perf_counter() - start\n        model.eval()\n        start = time.perf_counter()\n        val_n = 0\n        with torch.inference_mode():\n            for x, y in loaders["val"]:\n                x = x.cuda(non_blocking=True)\n                with torch.autocast(device_type="cuda", dtype=torch.float16):\n                    logits = model(x)\n                # Include CPU transfer used by the actual evaluation pipeline.\n                logits.float().cpu().numpy()\n                val_n += len(y)\n        torch.cuda.synchronize()\n        val_s = time.perf_counter() - start\n        row = {"backbone": name, "pretrained_cfg": cfg,\n               "train_images_seen": train_n, "val_images_seen": val_n,\n               "setup_s": setup_s, "train_s": train_s, "val_s": val_s,\n               "epoch_s": train_s + val_s,\n               "peak_allocated_mib": torch.cuda.max_memory_allocated() / 2**20}\n        report["models"].append(row)\n        write_json(args.out, report)  # Preserve completed models if the next one fails.\n        print(f"{name}: train={train_s:.1f}s, val={val_s:.1f}s, epoch={train_s + val_s:.1f}s", flush=True)\n        del model, optimizer, scheduler, scaler, groups, loaders, x, y, loss, logits\n        gc.collect()\n        torch.cuda.empty_cache()\n    print("Saved:", args.out)\n\n\ndef plan(args):\n    profile = json.loads(Path(args.profile).read_text(encoding="utf-8"))\n    rows = profile["models"]\n    times = {r["backbone"]: float(r["epoch_s"]) for r in rows}\n    if len(times) < 5:\n        raise ValueError("Measure at least five distinct backbones before planning.")\n    if any(not math.isfinite(t) or t <= 0 for t in times.values()):\n        raise ValueError("Epoch timings must be finite and positive.")\n    if not set(args.selected) <= times.keys():\n        raise ValueError("Every ablation backbone must have a measured epoch.")\n    if len(set(args.selected)) != len(args.selected):\n        raise ValueError("Ablation backbones must be distinct.")\n    final = args.final_backbone or args.selected[0]\n    if final not in args.selected:\n        raise ValueError("Final backbone must belong to --selected.")\n    epoch_h = lambda names: sum(times[n] for n in names) * args.epochs / 3600\n    # 7 fresh runs per ablation backbone: scratch, frozen, color, CutMix,\n    # label smoothing, focal, plus one combination. T00 is the B-stage recipe.\n    stages = [\n        ("B: backbone", len(times), epoch_h(list(times))),\n        ("T: 6 ablations + 1 combination / backbone", 7 * len(args.selected),\n         7 * epoch_h(args.selected)),\n        ("F: final 3 seeds + baseline 3 seeds", 6, 6 * epoch_h([final])),\n    ]\n    profiling_h = sum(float(r["setup_s"]) + times[r["backbone"]] for r in rows) / 3600\n    training_h = sum(s[2] for s in stages)\n    future_h = (training_h + args.extra_hours) * (1 + args.reserve)\n    total_h = profiling_h + future_h\n    lines = ["# Kế hoạch ngân sách GPU", "",\n             "Ước lượng từ một epoch train + val; chưa phải thời gian chạy toàn bộ thí nghiệm.",\n             "Chọn backbone trong kế hoạch chỉ để tính ngân sách. Quyết định chất lượng phải dựa trên macro-F1 val.",\n             "", f"GPU đã đo: {profile[\'gpu\']}. Batch: {profile[\'batch_size\']}; kích thước: {profile[\'img_size\']}; AMP: bật.",\n             f"Epoch mỗi lần chạy: {args.epochs}. Backbone ablation dự kiến: {\', \'.join(args.selected)}.",\n             "", "| Bước | Lần huấn luyện | Giờ GPU ước lượng |", "|---|---:|---:|"]\n    lines += [f"| {label} | {count} | {hours:.2f} |" for label, count, hours in stages]\n    lines += ["", f"Tổng số lần huấn luyện mới: {sum(s[1] for s in stages)} (ngoài profiling).",\n              "Không tái sử dụng profiling 1 epoch làm kết quả backbone 10–15 epoch.",\n              f"Profiling đã dùng: {profiling_h:.2f} giờ (gồm tải/khởi tạo model).",\n              f"Khoản thêm cho EDA, suy luận, test, benchmark: {args.extra_hours:.2f} giờ, là khoản dự trù chưa đo.",\n              f"Dự phòng: {args.reserve:.0%}; GPU còn cần: {future_h:.2f} giờ; tổng kể cả profiling: {total_h:.2f} giờ."]\n    if args.gpu_hours is None:\n        lines += ["", "**Chưa chốt kế hoạch:** chưa nhập hạn mức GPU. Chạy lại `plan --gpu-hours ...` sau khi kiểm tra tài khoản."]\n    else:\n        fits = total_h <= args.gpu_hours\n        lines += ["", f"Hạn mức tổng đã nhập: {args.gpu_hours:.2f} giờ; "\n                  + ("**ước lượng vừa ngân sách**." if fits else "**ước lượng vượt ngân sách**."),\n                  "Hạn mức này gồm profiling; dùng số giờ cùng một chu kỳ hạn mức."]\n        if not fits:\n            lines += ["Giảm 15 → 10 epoch, ablation trên 1 backbone, giữ 3 seed cho final và baseline; "\n                      "nếu hạ độ phân giải/batch hoặc đổi GPU thì đo lại toàn bộ backbone."]\n    lines += ["", "T00 dùng lại công thức của B-stage trên backbone được chọn; vòng cuối vẫn tính 3 lần baseline mới.",\n              "Seed sàng lọc: 0. Seed vòng cuối: 0, 1, 2. Mỗi ablation chỉ đổi một yếu tố.",\n              "Các lần sàng lọc không đánh giá test. Sau khi chốt trên val, chạy test một lần cho mỗi cấu hình/seed.",\n              "Cần đo lại nếu pipeline cuối khác profiler; chi phí CutMix/focal/EMA có thể khác công thức nền."]\n    Path(args.out).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.out).write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print("\\n".join(lines))\n\n\ndef positive_int(value):\n    n = int(value)\n    if n <= 0:\n        raise argparse.ArgumentTypeError("Must be positive")\n    return n\n\n\ndef nonnegative(value):\n    n = float(value)\n    if not math.isfinite(n) or n < 0:\n        raise argparse.ArgumentTypeError("Must be finite and nonnegative")\n    return n\n\n\ndef main():\n    if hasattr(sys.stdout, "reconfigure"):\n        sys.stdout.reconfigure(encoding="utf-8")\n    parser = argparse.ArgumentParser(description=__doc__)\n    commands = parser.add_subparsers(dest="command", required=True)\n    m = commands.add_parser("measure")\n    m.add_argument("--images-dir", default="data/images")\n    m.add_argument("--labels-dir", default="data/labels")\n    m.add_argument("--backbones", nargs="+", default=BACKBONES)\n    m.add_argument("--batch-size", type=positive_int, default=32)\n    m.add_argument("--img-size", type=positive_int, default=224)\n    m.add_argument("--workers", type=positive_int, default=2)\n    m.add_argument("--epochs", type=positive_int, default=10)\n    m.add_argument("--seed", type=int, default=0)\n    m.add_argument("--out", default="runs/gpu_budget/profile.json")\n    m.set_defaults(func=measure)\n    p = commands.add_parser("plan")\n    p.add_argument("--profile", default="runs/gpu_budget/profile.json")\n    p.add_argument("--selected", nargs="+", required=True)\n    p.add_argument("--final-backbone")\n    p.add_argument("--epochs", type=positive_int, default=10)\n    p.add_argument("--gpu-hours", type=nonnegative)\n    p.add_argument("--reserve", type=nonnegative, default=0.25)\n    p.add_argument("--extra-hours", type=nonnegative, default=1.0)\n    p.add_argument("--out", default="runs/gpu_budget/plan.md")\n    p.set_defaults(func=plan)\n    args = parser.parse_args()\n    args.func(args)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Đã tạo script đo thời gian.')


In [ ]:
import hashlib, urllib.request, zipfile
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)
archive = DATA_DIR / 'images.zip'
expected_md5 = 'b7b30f96d466fba86016aa5a26606e0f'
if not archive.exists():
    print('Đang tải ảnh DeepWeeds khoảng 490 MB...', flush=True)
    temporary = DATA_DIR / 'images.zip.part'
    urllib.request.urlretrieve('https://zenodo.org/records/7939060/files/images.zip?download=1', temporary)
    temporary.replace(archive)
h = hashlib.md5()
with archive.open('rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
assert h.hexdigest() == expected_md5, 'MD5 sai; xóa file images.zip lỗi rồi chạy lại ô này.'
print('MD5 đúng; đang giải nén...', flush=True)
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (DATA_DIR / info.filename).resolve()
        assert target.is_relative_to(DATA_DIR.resolve()), 'Đường dẫn không hợp lệ trong ZIP.'
    z.extractall(DATA_DIR)
# Xác định thư mục ảnh từ CSV, không giả định cấu trúc bên trong ZIP.
labels_url = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    urllib.request.urlretrieve(f'{labels_url}/{name}.csv', LABELS_DIR / f'{name}.csv')
import csv
with (LABELS_DIR / 'train_subset0.csv').open(encoding='utf-8-sig', newline='') as f:
    first_image = next(csv.DictReader(f))['Filename']
candidates = list(DATA_DIR.rglob(first_image))
assert len(candidates) == 1, 'Không xác định được duy nhất thư mục ảnh.'
IMAGES_DIR = candidates[0].parent
print('Ảnh:', IMAGES_DIR)
print('CSV fold 0:', LABELS_DIR)


In [ ]:
# Khoảng 5 epoch thực: một epoch mỗi backbone. Không có số giả định.
subprocess.run([sys.executable, str(REPO_DIR / 'tools/gpu_budget.py'), 'measure',
    '--images-dir', str(IMAGES_DIR), '--labels-dir', str(LABELS_DIR),
    '--batch-size', str(BATCH_SIZE), '--epochs', str(EPOCHS),
    '--out', str(OUTPUT_DIR / 'profile.json')], check=True)

In [ ]:
# Có thể chạy lại ô này để so kế hoạch 1 và 2 backbone, không cần đo lại.
command = [sys.executable, str(REPO_DIR / 'tools/gpu_budget.py'), 'plan',
    '--profile', str(OUTPUT_DIR / 'profile.json'), '--selected', *SELECTED,
    '--final-backbone', FINAL_BACKBONE, '--epochs', str(EPOCHS),
    '--reserve', '0.25', '--extra-hours', '1', '--out', str(OUTPUT_DIR / 'plan.md')]
if GPU_HOURS is not None:
    command += ['--gpu-hours', str(GPU_HOURS)]
subprocess.run(command, check=True)
print('Giữ lại profile.json và plan.md trong:', OUTPUT_DIR)

## Giữ lại kết quả
Trong thanh Files, tải về `gpu_budget/profile.json` và `gpu_budget/plan.md`.
Nếu OOM, giảm BATCH_SIZE xuống 16 và chạy lại từ ô cấu hình để đo cùng batch cho cả 5 backbone.
Có thể đổi SELECTED/GPU_HOURS rồi chỉ chạy lại ô tính ngân sách.
Nếu đổi GPU, batch hoặc độ phân giải thì phải đo lại.
